In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

print("Available Kaggle inputs:")
for path in INPUT_ROOT.iterdir():
    print(path)

In [ ]:
for path in INPUT_ROOT.rglob("*"):
    if path.is_file():
        print(path)

In [ ]:
from pathlib import Path

input_dirs = [p for p in Path("/kaggle/input").iterdir() if p.is_dir()]

assert input_dirs, "No Kaggle dataset found."

DATASET_ROOT = input_dirs[0]

print("Using dataset root:")
print(DATASET_ROOT)

In [ ]:
!pip install -q -U transformers datasets accelerate sentencepiece peft

In [ ]:
!pip install -q bitsandbytes

In [ ]:
jsonl_files = list(DATASET_ROOT.rglob("*.jsonl"))

print(f"Found {len(jsonl_files)} JSONL files:\n")

for path in jsonl_files:
    print(path)

In [ ]:
hindi_files = [
    p for p in jsonl_files
    if "hindi" in str(p).lower()
]

nepali_files = [
    p for p in jsonl_files
    if "nepali" in str(p).lower()
]

print("Hindi files:")
for p in hindi_files:
    print(p)

print("\nNepali files:")
for p in nepali_files:
    print(p)

In [ ]:
import json

for path in jsonl_files:
    print("\nFILE:", path)

    with open(path, encoding="utf-8") as f:
        first_line = next((line for line in f if line.strip()), None)

    if first_line:
        sample = json.loads(first_line)
        print(json.dumps(sample, ensure_ascii=False, indent=2)[:2000])

In [ ]:
from pathlib import Path
import shutil

WORK_ROOT = Path("/kaggle/working/reasoning_project")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

print(WORK_ROOT)

In [ ]:
for source in jsonl_files:
    destination = WORK_ROOT / source.name
    shutil.copy2(source, destination)
    print("Copied:", source.name)

In [ ]:
from collections import Counter

for path in WORK_ROOT.glob("*.jsonl"):
    count = 0
    categories = Counter()

    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            row = json.loads(line)
            count += 1

            if "category" in row:
                categories[row["category"]] += 1

    print(f"\n{path.name}")
    print("Rows:", count)
    print("Categories:", dict(categories))

In [ ]:
import json
import random
from pathlib import Path

random.seed(42)

SPLIT_ROOT = WORK_ROOT / "splits"
SPLIT_ROOT.mkdir(parents=True, exist_ok=True)

def split_jsonl(
    input_path,
    language,
    train_ratio=0.80,
    validation_ratio=0.10,
    test_ratio=0.10,
):
    with open(input_path, encoding="utf-8") as f:
        rows = [
            json.loads(line)
            for line in f
            if line.strip()
        ]

    random.shuffle(rows)

    total = len(rows)
    train_end = int(total * train_ratio)
    validation_end = train_end + int(total * validation_ratio)

    train_rows = rows[:train_end]
    validation_rows = rows[train_end:validation_end]
    test_rows = rows[validation_end:]

    language_root = SPLIT_ROOT / language
    language_root.mkdir(parents=True, exist_ok=True)

    outputs = {
        "train": train_rows,
        "validation": validation_rows,
        "test": test_rows,
    }

    for split_name, split_rows in outputs.items():
        output_path = language_root / f"{split_name}.jsonl"

        with open(output_path, "w", encoding="utf-8") as f:
            for row in split_rows:
                f.write(
                    json.dumps(
                        row,
                        ensure_ascii=False
                    ) + "\n"
                )

        print(
            f"{language} {split_name}: "
            f"{len(split_rows)} rows → {output_path}"
        )

    return outputs

In [ ]:
hindi_dedup = next(
    p for p in WORK_ROOT.glob("*")
    if "hindi" in p.name.lower()
    and "deduplicated" in p.name.lower()
)

nepali_dedup = next(
    p for p in WORK_ROOT.glob("*")
    if "nepali" in p.name.lower()
    and "deduplicated" in p.name.lower()
)

print(hindi_dedup)
print(nepali_dedup)

In [ ]:
hindi_splits = split_jsonl(
    hindi_dedup,
    language="hindi"
)

nepali_splits = split_jsonl(
    nepali_dedup,
    language="nepali"
)

In [ ]:
def count_rows(path):
    with open(path, encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

for path in sorted(SPLIT_ROOT.rglob("*.jsonl")):
    print(path.relative_to(SPLIT_ROOT), count_rows(path))

In [ ]:
from pathlib import Path

all_files = list(DATASET_ROOT.rglob("*"))

model_files = [
    p for p in all_files
    if p.is_file()
    and p.suffix.lower() in {
        ".json", ".safetensors", ".bin", ".model", ".txt"
    }
]

for path in model_files:
    print(path)

In [ ]:
config_paths = list(DATASET_ROOT.rglob("config.json"))

print("Config files found:")
for path in config_paths:
    print(path)

In [ ]:
import json
from pathlib import Path

SPLIT_ROOT = Path("/kaggle/working/reasoning_project/splits")

HINDI_TRAIN = SPLIT_ROOT / "hindi/train.jsonl"
HINDI_VAL = SPLIT_ROOT / "hindi/validation.jsonl"

NEPALI_TRAIN = SPLIT_ROOT / "nepali/train.jsonl"
NEPALI_VAL = SPLIT_ROOT / "nepali/validation.jsonl"


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]


hindi_train = load_jsonl(HINDI_TRAIN)
hindi_val = load_jsonl(HINDI_VAL)

nepali_train = load_jsonl(NEPALI_TRAIN)
nepali_val = load_jsonl(NEPALI_VAL)

print("Hindi train:", len(hindi_train))
print("Hindi validation:", len(hindi_val))
print("Nepali train:", len(nepali_train))
print("Nepali validation:", len(nepali_val))

In [ ]:
from pprint import pprint

print("HINDI SAMPLE")
pprint(hindi_train[0], sort_dicts=False)

print("\nNEPALI SAMPLE")
pprint(nepali_train[0], sort_dicts=False)

In [ ]:
def format_example(row, language):
    language_name = {
        "hindi": "Hindi",
        "nepali": "Nepali",
    }[language]

    question = row.get("question", "").strip()
    reasoning = row.get("reasoning", "").strip()
    answer = row.get("answer", "").strip()

    return (
        f"### भाषा\n"
        f"{language_name}\n\n"
        f"### प्रश्न\n"
        f"{question}\n\n"
        f"### तर्क\n"
        f"{reasoning}\n\n"
        f"### उत्तर\n"
        f"{answer}"
    )


In [ ]:
print(format_example(hindi_train[0], "hindi"))
print("\n" + "=" * 80 + "\n")
print(format_example(nepali_train[0], "nepali"))

In [ ]:
from datasets import Dataset, DatasetDict


def safe_string(value):
    if value is None:
        return ""
    return str(value)


def make_dataset_row(row, language):
    return {
        "text": format_example(row, language),
        "language": safe_string(language),
        "category": safe_string(row.get("category")),
        "id": safe_string(row.get("id")),
    }


train_rows = (
    [
        make_dataset_row(row, "hindi")
        for row in hindi_train
    ]
    +
    [
        make_dataset_row(row, "nepali")
        for row in nepali_train
    ]
)

validation_rows = (
    [
        make_dataset_row(row, "hindi")
        for row in hindi_val
    ]
    +
    [
        make_dataset_row(row, "nepali")
        for row in nepali_val
    ]
)

dataset = DatasetDict({
    "train": Dataset.from_list(train_rows),
    "validation": Dataset.from_list(validation_rows),
})

print(dataset)
print(dataset["train"][0])

In [ ]:
from datasets import Dataset, DatasetDict

train_texts = (
    [
        {"text": format_example(row, "hindi")}
        for row in hindi_train
    ]
    +
    [
        {"text": format_example(row, "nepali")}
        for row in nepali_train
    ]
)

validation_texts = (
    [
        {"text": format_example(row, "hindi")}
        for row in hindi_val
    ]
    +
    [
        {"text": format_example(row, "nepali")}
        for row in nepali_val
    ]
)

dataset = DatasetDict({
    "train": Dataset.from_list(train_texts),
    "validation": Dataset.from_list(validation_texts),
})

print(dataset)
print(dataset["train"][0]["text"])

In [ ]:
from pathlib import Path

MODEL_ROOT = Path(
    "/kaggle/input/datasets/"
    "bharadwajvss/trained-language-model"
)

HINDI_MODEL_PATH = MODEL_ROOT / "hindi_model.pt"
NEPALI_MODEL_PATH = MODEL_ROOT / "nepal_model.pt"

print(HINDI_MODEL_PATH.exists(), HINDI_MODEL_PATH)
print(NEPALI_MODEL_PATH.exists(), NEPALI_MODEL_PATH)

In [ ]:
import torch

hindi_checkpoint = torch.load(
    HINDI_MODEL_PATH,
    map_location="cpu",
    weights_only=False,
)

nepali_checkpoint = torch.load(
    NEPALI_MODEL_PATH,
    map_location="cpu",
    weights_only=False,
)

print("Hindi checkpoint type:", type(hindi_checkpoint))
print("Nepali checkpoint type:", type(nepali_checkpoint))

In [ ]:
def inspect_checkpoint(checkpoint, name):
    print(f"\n===== {name} =====")

    if isinstance(checkpoint, dict):
        print("Keys:")
        for key in checkpoint.keys():
            print(" ", key)

        for key, value in checkpoint.items():
            print(
                f"\n{key}: "
                f"type={type(value)}"
            )

            if isinstance(value, dict):
                print("  nested keys:", list(value.keys())[:20])

    else:
        print("Object:", checkpoint)


inspect_checkpoint(hindi_checkpoint, "HINDI")
inspect_checkpoint(nepali_checkpoint, "NEPALI")

In [ ]:
def inspect_state_dict(checkpoint, name):
    print(f"\n===== {name} =====")

    if isinstance(checkpoint, dict):
        possible_keys = [
            "state_dict",
            "model_state_dict",
            "model",
            "weights",
        ]

        for key in possible_keys:
            if key in checkpoint:
                value = checkpoint[key]

                if isinstance(value, dict):
                    print(f"Found `{key}` with {len(value)} entries")
                    print("First keys:")

                    for item in list(value.keys())[:20]:
                        tensor = value[item]
                        shape = getattr(tensor, "shape", None)
                        print(" ", item, shape)

                    return key

        tensor_items = {
            key: value
            for key, value in checkpoint.items()
            if torch.is_tensor(value)
        }

        if tensor_items:
            print(
                "Checkpoint itself appears to be a state_dict "
                f"with {len(tensor_items)} tensors."
            )

            for key, tensor in list(tensor_items.items())[:20]:
                print(" ", key, tensor.shape)

            return "direct_state_dict"

    print("Could not identify a state dictionary.")
    return None


hindi_state_key = inspect_state_dict(
    hindi_checkpoint,
    "HINDI"
)

nepali_state_key = inspect_state_dict(
    nepali_checkpoint,
    "NEPALI"
)

In [ ]:
for name, checkpoint in [
    ("Hindi", hindi_checkpoint),
    ("Nepali", nepali_checkpoint),
]:
    print(f"\n===== {name} metadata =====")

    if isinstance(checkpoint, dict):
        for key in checkpoint.keys():
            lower_key = str(key).lower()

            if any(word in lower_key for word in [
                "vocab",
                "token",
                "stoi",
                "itos",
                "word",
                "config",
                "args",
                "embedding",
            ]):
                print("Possible relevant key:", key)

In [ ]:
print("HINDI CONFIG")
for key, value in hindi_checkpoint["config"].items():
    print(f"{key}: {value}")

print("\nNEPALI CONFIG")
for key, value in nepali_checkpoint["config"].items():
    print(f"{key}: {value}")

In [ ]:
print("\nHindi state dict keys:")
for key in hindi_checkpoint["model_state_dict"].keys():
    print(key)

In [ ]:
for language, checkpoint in [
    ("Hindi", hindi_checkpoint),
    ("Nepali", nepali_checkpoint),
]:
    print(f"\n===== {language} checkpoint metadata =====")

    for key, value in checkpoint.items():
        if key not in {
            "model_state_dict",
            "optimizer_state_dict",
            "scheduler_state_dict",
            "scaler_state_dict",
        }:
            print(f"{key}: {value}")

In [ ]:
for language, checkpoint in [
    ("Hindi", hindi_checkpoint),
    ("Nepali", nepali_checkpoint),
]:
    print(f"\n{language}")

    for key, value in checkpoint.items():
        print(
            key,
            "is tensor:", torch.is_tensor(value),
            "type:", type(value)
        )

In [ ]:
print("Hindi config:")
print(hindi_checkpoint["config"])

print("\nNepali config:")
print(nepali_checkpoint["config"])

In [ ]:
print("===== ALL HINDI MODEL KEYS =====")

for key, value in hindi_checkpoint["model_state_dict"].items():
    print(key, tuple(value.shape))

In [ ]:
print("\n===== ALL NEPALI MODEL KEYS =====")

for key, value in nepali_checkpoint["model_state_dict"].items():
    print(key, tuple(value.shape))

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()

        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.d_model = d_model

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        # Causal attention: tokens cannot see future tokens
        attention_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device),
            diagonal=1
        ).bool()

        attention_scores = attention_scores.masked_fill(
            causal_mask,
            float("-inf")
        )

        attention_weights = F.softmax(attention_scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        output = attention_weights @ v

        output = output.transpose(1, 2).contiguous()
        output = output.view(batch_size, seq_len, d_model)

        return self.out_proj(output)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)

        self.attn = CausalSelfAttention(
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout
        )

        self.ln2 = nn.LayerNorm(d_model)

        # Important: checkpoint expects ff.net.0 and ff.net.3
        self.ff = FeedForward(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.ln1(x)))
        x = x + self.dropout(self.ff(self.ln2(x)))
        return x

class ReasoningTransformer(nn.Module):
    def __init__(self, config, final_norm_name="final_norm"):
        super().__init__()

        self.vocab_size = config["vocab_size"]
        self.d_model = config["d_model"]
        self.seq_len = config["seq_len"]

        self.token_embedding = nn.Embedding(
            self.vocab_size,
            self.d_model
        )

        self.position_embedding = nn.Embedding(
            self.seq_len,
            self.d_model
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config["d_model"],
                num_heads=config["num_heads"],
                d_ff=config["d_ff"],
                dropout=config["dropout"]
            )
            for _ in range(config["num_layers"])
        ])

        # The checkpoint uses different names for Hindi and Nepali.
        if final_norm_name == "final_norm":
            self.final_norm = nn.LayerNorm(self.d_model)
        else:
            self.ln_f = nn.LayerNorm(self.d_model)

        self.lm_head = nn.Linear(
            self.d_model,
            self.vocab_size
        )

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape

        if seq_len > self.seq_len:
            raise ValueError(
                f"Sequence length {seq_len} exceeds maximum {self.seq_len}"
            )

        positions = torch.arange(
            seq_len,
            device=input_ids.device
        )

        x = self.token_embedding(input_ids)
        x = x + self.position_embedding(positions)

        for block in self.blocks:
            x = block(x)

        if hasattr(self, "final_norm"):
            x = self.final_norm(x)
        else:
            x = self.ln_f(x)

        logits = self.lm_head(x)

        return logits

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

hindi_config = hindi_checkpoint["config"]
nepali_config = nepali_checkpoint["config"]

hindi_model = ReasoningTransformer(
    hindi_config,
    final_norm_name="final_norm"
).to(device)

nepali_model = ReasoningTransformer(
    nepali_config,
    final_norm_name="ln_f"
).to(device)

hindi_model.load_state_dict(
    hindi_checkpoint["model_state_dict"],
    strict=True
)

nepali_model.load_state_dict(
    nepali_checkpoint["model_state_dict"],
    strict=True
)

hindi_model.eval()
nepali_model.eval()

print("Hindi model loaded successfully")
print("Nepali model loaded successfully")

In [ ]:
import os

print("===== TOKENIZER / VOCAB FILES =====")

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if any(term in file.lower() for term in [
            "tokenizer",
            "vocab",
            "sentencepiece",
            "spiece",
            "merges",
            "token"
        ]):
            print(os.path.join(root, file))

print("\n===== PYTHON / CONFIG FILES IN WORKING =====")

for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        if file.endswith((".py", ".json", ".yaml", ".yml", ".txt", ".model")):
            print(os.path.join(root, file))

In [ ]:
import os

DATA_ROOT = "/kaggle/input/datasets/bharadwajvss/trained-language-model"

print("Files in dataset:")
for file in os.listdir(DATA_ROOT):
    print(file)

In [ ]:
import json
import os

DATA_ROOT = "/kaggle/input/datasets/bharadwajvss/trained-language-model"

path = os.path.join(
    DATA_ROOT,
    "hindi_reasoning_deduplicated.jsonl"
)

with open(path, "r", encoding="utf-8") as f:
    first_line = json.loads(next(f))

print("First record:")
print(first_line)

print("\nFields:")
for key, value in first_line.items():
    print(
        key,
        "->",
        type(value).__name__,
        "->",
        repr(value)[:500]
    )

In [ ]:
print("\nAll keys from first 20 records:")

all_keys = set()

with open(path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 20:
            break

        record = json.loads(line)
        all_keys.update(record.keys())

print(all_keys)

In [ ]:
for root, dirs, files in os.walk("/kaggle/working"):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import os
from pathlib import Path

print("Searching all Kaggle datasets for tokenizer files...\n")

candidates = []

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        name = file.lower()

        if (
            "tokenizer" in name
            or "sentencepiece" in name
            or "spiece" in name
            or "vocab" in name
            or file.endswith((
                ".model",
                ".model.vocab",
                ".json",
                ".txt"
            ))
        ):
            path = os.path.join(root, file)

            # Skip the reasoning JSONL files
            if not file.endswith(".jsonl"):
                candidates.append(path)

for path in sorted(candidates):
    print(path)

print("\nTotal possible files:", len(candidates))

In [ ]:
for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)

    if level <= 4:
        print("\n", root)
        for file in files:
            print("   ", file)

In [ ]:
import json

HINDI_TOKENIZER_PATH = (
    "/kaggle/input/datasets/bharadwajvss/"
    "hindi-phase2-final/hindi_bpe_31744.json"
)

with open(HINDI_TOKENIZER_PATH, "r", encoding="utf-8") as f:
    raw = f.read()

print(raw[:2000])

In [ ]:
try:
    data = json.loads(raw)
    print("Valid JSON")
    print(type(data))
    print(data.keys() if isinstance(data, dict) else data[:3])
except Exception as e:
    print("JSON parsing failed:", e)
    print("Problem near beginning:")
    print(repr(raw[:300]))

In [ ]:
import re
import json

def load_bpe_file(path):
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # Some custom files use Python-style single quotes
        repaired = raw.replace("'", '"')
        return json.loads(repaired)


hindi_bpe = load_bpe_file(HINDI_TOKENIZER_PATH)

print(type(hindi_bpe))

if isinstance(hindi_bpe, dict):
    print("Keys:", list(hindi_bpe.keys())[:30])
else:
    print("Length:", len(hindi_bpe))

In [ ]:
NEPALI_TOKENIZER_PATH = (
    "/kaggle/input/datasets/bharadwajvss/"
    "nepali-phase2-source/nepali_bpe_31744.json"
)

with open(NEPALI_TOKENIZER_PATH, "r", encoding="utf-8") as f:
    print(f.read()[:2000])

In [ ]:
import json
import regex
from functools import lru_cache


class RegexBPE:
    def __init__(self, path):
        with open(path, "r", encoding="utf-8") as f:
            config = json.load(f)

        self.config = config
        self.vocab_size = config["vocab_size"]
        self.base_vocab_size = config["base_vocab_size"]
        self.pattern = regex.compile(config["pattern"])

        # Each merge is: [left_id, right_id, new_id]
        self.merges = config["merges"]

        self.merge_rank = {}
        self.token_bytes = {
            i: bytes([i])
            for i in range(self.base_vocab_size)
        }

        for rank, merge in enumerate(self.merges):
            left, right, new_id = merge

            self.merge_rank[(left, right)] = rank

            self.token_bytes[new_id] = (
                self.token_bytes[left]
                + self.token_bytes[right]
            )

    def _encode_piece(self, piece):
        ids = list(piece.encode("utf-8"))

        while len(ids) >= 2:
            best_pair = None
            best_rank = None

            for i in range(len(ids) - 1):
                pair = (ids[i], ids[i + 1])

                if pair in self.merge_rank:
                    rank = self.merge_rank[pair]

                    if best_rank is None or rank < best_rank:
                        best_rank = rank
                        best_pair = pair

            if best_pair is None:
                break

            new_id = None

            # Find the resulting token ID for this pair
            for left, right, merged_id in self.merges:
                if left == best_pair[0] and right == best_pair[1]:
                    new_id = merged_id
                    break

            if new_id is None:
                break

            new_ids = []
            i = 0

            while i < len(ids):
                if (
                    i < len(ids) - 1
                    and ids[i] == best_pair[0]
                    and ids[i + 1] == best_pair[1]
                ):
                    new_ids.append(new_id)
                    i += 2
                else:
                    new_ids.append(ids[i])
                    i += 1

            ids = new_ids

        return ids

    def encode(self, text):
        token_ids = []

        for piece in self.pattern.findall(text):
            token_ids.extend(self._encode_piece(piece))

        return token_ids

    def decode(self, ids):
        output = bytearray()

        for token_id in ids:
            if token_id not in self.token_bytes:
                raise ValueError(f"Unknown token ID: {token_id}")

            output.extend(self.token_bytes[token_id])

        return output.decode("utf-8", errors="replace")

In [ ]:
HINDI_BPE_PATH = (
    "/kaggle/input/datasets/bharadwajvss/"
    "hindi-phase2-final/hindi_bpe_31744.json"
)

NEPALI_BPE_PATH = (
    "/kaggle/input/datasets/bharadwajvss/"
    "nepali-phase2-source/nepali_bpe_31744.json"
)

hindi_tokenizer = RegexBPE(HINDI_BPE_PATH)
nepali_tokenizer = RegexBPE(NEPALI_BPE_PATH)

print("Hindi vocab:", hindi_tokenizer.vocab_size)
print("Nepali vocab:", nepali_tokenizer.vocab_size)

In [ ]:
hindi_text = "यदि रवि गीता से बड़ा है।"
nepali_text = "यदि राम सीताभन्दा ठूलो छ।"

hindi_ids = hindi_tokenizer.encode(hindi_text)
nepali_ids = nepali_tokenizer.encode(nepali_text)

print("Hindi IDs:", hindi_ids)
print("Hindi decoded:", hindi_tokenizer.decode(hindi_ids))

print("\nNepali IDs:", nepali_ids)
print("Nepali decoded:", nepali_tokenizer.decode(nepali_ids))

In [ ]:
@torch.no_grad()
def generate(
    model,
    tokenizer,
    prompt,
    max_new_tokens=100,
    temperature=0.7,
    top_k=40
):
    model.eval()

    token_ids = tokenizer.encode(prompt)

    input_ids = torch.tensor(
        [token_ids],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_new_tokens):
        input_cond = input_ids[:, -128:]

        logits = model(input_cond)
        next_token_logits = logits[:, -1, :]

        next_token_logits = next_token_logits / temperature

        if top_k is not None:
            values, indices = torch.topk(
                next_token_logits,
                min(top_k, next_token_logits.size(-1))
            )

            filtered_logits = torch.full_like(
                next_token_logits,
                float("-inf")
            )

            filtered_logits.scatter_(
                1,
                indices,
                values
            )

            next_token_logits = filtered_logits

        probabilities = F.softmax(
            next_token_logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probabilities,
            num_samples=1
        )

        input_ids = torch.cat(
            [input_ids, next_token],
            dim=1
        )

    return tokenizer.decode(input_ids[0].tolist())

In [ ]:
prompt = """प्रश्न: यदि रवि का मान गीता से अधिक है और गीता का मान विनय से अधिक है, तो इनमें सबसे बड़ा कौन है?

समाधान:"""

print(
    generate(
        hindi_model,
        hindi_tokenizer,
        prompt,
        max_new_tokens=80,
        temperature=0.7,
        top_k=40
    )
)

In [ ]:
prompt = """प्रश्न: यदि रामको मान सीताभन्दा ठूलो छ र सीताको मान हरिभन्दा ठूलो छ भने सबैभन्दा ठूलो को हो?

समाधान:"""

print(
    generate(
        nepali_model,
        nepali_tokenizer,
        prompt,
        max_new_tokens=80,
        temperature=0.7,
        top_k=40
    )
)

In [ ]:
@torch.no_grad()
def generate_greedy(model, tokenizer, prompt, max_new_tokens=80):
    model.eval()

    ids = tokenizer.encode(prompt)

    input_ids = torch.tensor(
        [ids],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_new_tokens):
        input_cond = input_ids[:, -128:]

        logits = model(input_cond)
        next_token = torch.argmax(
            logits[:, -1, :],
            dim=-1,
            keepdim=True
        )

        input_ids = torch.cat(
            [input_ids, next_token],
            dim=1
        )

    return tokenizer.decode(input_ids[0].tolist())

In [ ]:
prompt = """प्रश्न: यदि रवि का मान गीता से अधिक है और गीता का मान विनय से अधिक है, तो इनमें सबसे बड़ा कौन है?

समाधान:"""

print(
    generate_greedy(
        hindi_model,
        hindi_tokenizer,
        prompt,
        max_new_tokens=60
    )
)

In [ ]:
prompt = """प्रश्न: यदि रवि का मान गीता से अधिक है और गीता का मान विनय से अधिक है, तो इनमें सबसे बड़ा कौन है?

समाधान: रवि > गीता और गीता > विनय। इसलिए संक्रमण नियम के अनुसार रवि > विनय और रवि सबसे बड़ा है।

उत्तर:"""

print(
    generate_greedy(
        hindi_model,
        hindi_tokenizer,
        prompt,
        max_new_tokens=20
    )
)

In [ ]:
prompt = """प्रश्न: यदि रामको मान सीताभन्दा ठूलो छ र सीताको मान हरिभन्दा ठूलो छ भने सबैभन्दा ठूलो को हो?

समाधान: राम > सीता र सीता > हरि। त्यसैले राम सबैभन्दा ठूलो हो।

उत्तर:"""

print(
    generate_greedy(
        nepali_model,
        nepali_tokenizer,
        prompt,
        max_new_tokens=20
    )
)

In [ ]:
tests = [
    "प्रश्न: यदि रवि का मान गीता से अधिक है।",
    "समाधान: रवि सबसे बड़ा है।",
    "उत्तर: रवि",
    "यदि राम सीताभन्दा ठूलो छ।",
]

for text in tests:
    ids = hindi_tokenizer.encode(text)
    decoded = hindi_tokenizer.decode(ids)

    print("Original:", repr(text))
    print("Decoded :", repr(decoded))
    print("Match   :", text == decoded)
    print("-" * 50)

In [ ]:
import json

train_path = (
    "/kaggle/working/reasoning_project/"
    "splits/hindi/train.jsonl"
)

with open(train_path, "r", encoding="utf-8") as f:
    example = json.loads(next(f))

print(example["text"])

print("\nMODEL OUTPUT:\n")

print(
    generate_greedy(
        hindi_model,
        hindi_tokenizer,
        example["text"][:250],
        max_new_tokens=60
    )
)

In [ ]:
import torch

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

print("Hindi parameters:", count_parameters(hindi_model))
print("Nepali parameters:", count_parameters(nepali_model))

print("Hindi device:", next(hindi_model.parameters()).device)
print("Nepali device:", next(nepali_model.parameters()).device)

print("Hindi vocab:", hindi_tokenizer.vocab_size)
print("Nepali vocab:", nepali_tokenizer.vocab_size)

print("Hindi checkpoint step:", hindi_checkpoint["step"])
print("Nepali checkpoint step:", nepali_checkpoint["step"])

print("\nPHASE 2 CHECKPOINT VALIDATION COMPLETE")

In [ ]:
import os
import json
import torch
from torch.utils.data import Dataset, DataLoader

SEQ_LEN = 128
BATCH_SIZE = 4
NUM_WORKERS = 2

SPLIT_DIR = "/kaggle/working/reasoning_project/splits"


class TokenStreamDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, seq_len=128):
        self.seq_len = seq_len
        self.tokenizer = tokenizer

        all_tokens = []

        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                record = json.loads(line)
                text = record["text"]

                token_ids = tokenizer.encode(text)

                # Separate examples slightly
                token_ids += tokenizer.encode("\n\n")

                all_tokens.extend(token_ids)

        # Need seq_len + 1 tokens for input/target shifting
        usable_length = ((len(all_tokens) - 1) // seq_len) * seq_len
        all_tokens = all_tokens[:usable_length + 1]

        self.tokens = torch.tensor(all_tokens, dtype=torch.long)

        self.num_samples = (len(self.tokens) - 1) // seq_len

        print(f"Loaded: {jsonl_path}")
        print(f"Total tokens: {len(self.tokens):,}")
        print(f"Training sequences: {self.num_samples:,}")

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        start = idx * self.seq_len
        end = start + self.seq_len + 1

        chunk = self.tokens[start:end]

        input_ids = chunk[:-1]
        labels = chunk[1:]

        return {
            "input_ids": input_ids,
            "labels": labels,
        }

In [ ]:
SPLIT_DIR = "/kaggle/working/reasoning_project/splits"

print(os.listdir(f"{SPLIT_DIR}/hindi"))
print(os.listdir(f"{SPLIT_DIR}/nepali"))

In [ ]:
hindi_train_path = f"{SPLIT_DIR}/hindi/train.jsonl"
hindi_val_path   = f"{SPLIT_DIR}/hindi/validation.jsonl"
hindi_test_path  = f"{SPLIT_DIR}/hindi/test.jsonl"

nepali_train_path = f"{SPLIT_DIR}/nepali/train.jsonl"
nepali_val_path   = f"{SPLIT_DIR}/nepali/validation.jsonl"
nepali_test_path  = f"{SPLIT_DIR}/nepali/test.jsonl"

In [ ]:
hindi_train_dataset = TokenStreamDataset(
    hindi_train_path, hindi_tokenizer, SEQ_LEN
)

hindi_val_dataset = TokenStreamDataset(
    hindi_val_path, hindi_tokenizer, SEQ_LEN
)

hindi_test_dataset = TokenStreamDataset(
    hindi_test_path, hindi_tokenizer, SEQ_LEN
)

nepali_train_dataset = TokenStreamDataset(
    nepali_train_path, nepali_tokenizer, SEQ_LEN
)

nepali_val_dataset = TokenStreamDataset(
    nepali_val_path, nepali_tokenizer, SEQ_LEN
)

nepali_test_dataset = TokenStreamDataset(
    nepali_test_path, nepali_tokenizer, SEQ_LEN
)

In [ ]:
hindi_train_loader = DataLoader(
    hindi_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

hindi_val_loader = DataLoader(
    hindi_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

hindi_test_loader = DataLoader(
    hindi_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

nepali_train_loader = DataLoader(
    nepali_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

nepali_val_loader = DataLoader(
    nepali_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

nepali_test_loader = DataLoader(
    nepali_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

batch = next(iter(hindi_train_loader))

print("Input shape:", batch["input_ids"].shape)
print("Labels shape:", batch["labels"].shape)
print("Input dtype:", batch["input_ids"].dtype)
print("Labels dtype:", batch["labels"].dtype)

print("\nPHASE 3.1 DATALOADERS READY")

In [ ]:
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
            labels = batch["labels"].to(DEVICE, non_blocking=True)

            logits = model(input_ids)

            # Support models returning either logits or (logits, loss)
            if isinstance(logits, tuple):
                logits = logits[0]

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1)
            )

            total_loss += loss.item()
            total_batches += 1

    model.train()
    return total_loss / max(total_batches, 1)

In [ ]:
def finetune_model(
    model,
    train_loader,
    val_loader,
    output_path,
    epochs=1,
    lr=1e-5,
    grad_accum_steps=4,
    max_grad_norm=1.0
):
    model = model.to(DEVICE)
    model.train()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=0.01
    )

    best_val_loss = float("inf")

    model_config = {
        "vocab_size": 31744,
        "d_model": 288,
        "num_heads": 8,
        "num_layers": 6,
        "d_ff": 1440,
        "seq_len": 128,
        "dropout": 0.1,
    }

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        running_loss = 0.0

        progress = tqdm(
            train_loader,
            desc=f"Epoch {epoch + 1}/{epochs}"
        )

        for step, batch in enumerate(progress):
            input_ids = batch["input_ids"].to(
                DEVICE,
                non_blocking=True
            )

            labels = batch["labels"].to(
                DEVICE,
                non_blocking=True
            )

            logits = model(input_ids)

            if isinstance(logits, tuple):
                logits = logits[0]

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1)
            )

            (loss / grad_accum_steps).backward()

            running_loss += loss.item()

            if (
                (step + 1) % grad_accum_steps == 0
                or (step + 1) == len(train_loader)
            ):
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_grad_norm
                )

                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            progress.set_postfix(
                loss=f"{loss.item():.4f}"
            )

        train_loss = running_loss / len(train_loader)
        val_loss = evaluate(model, val_loader)

        print(
            f"Epoch {epoch + 1}: "
            f"train_loss={train_loss:.4f}, "
            f"val_loss={val_loss:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "config": model_config,
                    "epoch": epoch + 1,
                    "best_val_loss": best_val_loss,
                },
                output_path
            )

            print(f"Saved best checkpoint: {output_path}")

    return best_val_loss

In [ ]:
hindi_output_path = (
    "/kaggle/working/reasoning_project/"
    "hindi_finetuned_phase3.pt"
)

nepali_output_path = (
    "/kaggle/working/reasoning_project/"
    "nepali_finetuned_phase3.pt"
)

print("Hindi output path:", hindi_output_path)
print("Nepali output path:", nepali_output_path)

In [ ]:
hindi_best_val_loss = finetune_model(
    model=hindi_model,
    train_loader=hindi_train_loader,
    val_loader=hindi_val_loader,
    output_path=hindi_output_path,
    epochs=1,
    lr=1e-5,
    grad_accum_steps=4
)

print("Hindi fine-tuning complete.")
print("Best validation loss:", hindi_best_val_loss)

In [ ]:
nepali_output_path = (
    "/kaggle/working/reasoning_project/"
    "nepali_finetuned_phase3.pt"
)

nepali_best_val_loss = finetune_model(
    model=nepali_model,
    train_loader=nepali_train_loader,
    val_loader=nepali_val_loader,
    output_path=nepali_output_path,
    epochs=1,
    lr=1e-5,
    grad_accum_steps=4
)

print("Nepali fine-tuning complete.")
print("Best validation loss:", nepali_best_val_loss)

In [ ]:
hindi_ft_checkpoint = torch.load(
    hindi_output_path,
    map_location=DEVICE,
    weights_only=False
)

nepali_ft_checkpoint = torch.load(
    nepali_output_path,
    map_location=DEVICE,
    weights_only=False
)

hindi_model.load_state_dict(
    hindi_ft_checkpoint["model_state_dict"]
)

nepali_model.load_state_dict(
    nepali_ft_checkpoint["model_state_dict"]
)

hindi_model.eval()
nepali_model.eval()

print("Hindi fine-tuned checkpoint loaded.")
print("Nepali fine-tuned checkpoint loaded.")

In [ ]:
@torch.no_grad()
def generate_text(
    model,
    tokenizer,
    prompt,
    max_new_tokens=100,
    temperature=0.7,
    top_k=50
):
    model.eval()

    input_ids = tokenizer.encode(prompt)
    input_ids = torch.tensor(
        [input_ids],
        dtype=torch.long,
        device=DEVICE
    )

    for _ in range(max_new_tokens):
        input_context = input_ids[:, -SEQ_LEN:]

        logits = model(input_context)

        if isinstance(logits, tuple):
            logits = logits[0]

        next_token_logits = logits[:, -1, :]

        next_token_logits = next_token_logits / temperature

        if top_k is not None:
            values, indices = torch.topk(
                next_token_logits,
                min(top_k, next_token_logits.size(-1))
            )

            filtered_logits = torch.full_like(
                next_token_logits,
                float("-inf")
            )

            filtered_logits.scatter_(
                1,
                indices,
                values
            )

            next_token_logits = filtered_logits

        probabilities = torch.softmax(
            next_token_logits,
            dim=-1
        )

        next_token = torch.multinomial(
            probabilities,
            num_samples=1
        )

        input_ids = torch.cat(
            [input_ids, next_token],
            dim=1
        )

    return tokenizer.decode(
        input_ids[0].tolist()
    )

In [ ]:
hindi_prompt = """प्रश्न: यदि रवि का मान गीता से अधिक है और गीता का मान विनय से अधिक है, तो इनमें सबसे बड़ा कौन है?

समाधान:"""

print(
    generate_text(
        hindi_model,
        hindi_tokenizer,
        hindi_prompt,
        max_new_tokens=80,
        temperature=0.5,
        top_k=30
    )
)

In [ ]:
nepali_prompt = """प्रश्न: यदि रामको अंक श्यामको भन्दा बढी छ र श्यामको अंक हरिको भन्दा बढी छ भने सबैभन्दा ठूलो अंक कसको हो?

समाधान:"""

print(
    generate_text(
        nepali_model,
        nepali_tokenizer,
        nepali_prompt,
        max_new_tokens=80,
        temperature=0.5,
        top_k=30
    )
)

In [ ]:
@torch.no_grad()
def generate_greedy(
    model,
    tokenizer,
    prompt,
    max_new_tokens=80
):
    model.eval()

    token_ids = tokenizer.encode(prompt)

    input_ids = torch.tensor(
        [token_ids],
        dtype=torch.long,
        device=DEVICE
    )

    for _ in range(max_new_tokens):
        context = input_ids[:, -SEQ_LEN:]

        logits = model(context)

        if isinstance(logits, tuple):
            logits = logits[0]

        next_token = torch.argmax(
            logits[:, -1, :],
            dim=-1,
            keepdim=True
        )

        input_ids = torch.cat(
            [input_ids, next_token],
            dim=1
        )

    return tokenizer.decode(
        input_ids[0].tolist()
    )

In [ ]:
hindi_prompt = """प्रश्न: यदि रवि का मान गीता से अधिक है और गीता का मान विनय से अधिक है, तो इनमें सबसे बड़ा कौन है?

समाधान:"""

print(
    generate_greedy(
        hindi_model,
        hindi_tokenizer,
        hindi_prompt,
        max_new_tokens=60
    )
)

In [ ]:
nepali_prompt = """प्रश्न: यदि रामको अंक श्यामको भन्दा बढी छ र श्यामको अंक हरिको भन्दा बढी छ भने सबैभन्दा ठूलो अंक कसको हो?

समाधान:"""

print(
    generate_greedy(
        nepali_model,
        nepali_tokenizer,
        nepali_prompt,
        max_new_tokens=60
    )
)

In [ ]:
class ExampleDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, seq_len=128):
        self.samples = []

        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                record = json.loads(line)
                text = record["text"].strip()

                token_ids = tokenizer.encode(text)

                # Keep only examples that fit the model context
                if len(token_ids) >= 2:
                    token_ids = token_ids[:seq_len + 1]
                    self.samples.append(token_ids)

        print(f"Loaded: {jsonl_path}")
        print(f"Examples: {len(self.samples):,}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        token_ids = self.samples[idx]

        input_ids = token_ids[:-1]
        labels = token_ids[1:]

        # Pad short examples with -100 labels
        # so padding does not contribute to loss
        pad_length = SEQ_LEN - len(input_ids)

        if pad_length > 0:
            input_ids = input_ids + [0] * pad_length
            labels = labels + [-100] * pad_length

        return {
            "input_ids": torch.tensor(
                input_ids,
                dtype=torch.long
            ),
            "labels": torch.tensor(
                labels,
                dtype=torch.long
            ),
        }

In [ ]:
hindi_train_dataset = ExampleDataset(
    hindi_train_path, hindi_tokenizer, SEQ_LEN
)

hindi_val_dataset = ExampleDataset(
    hindi_val_path, hindi_tokenizer, SEQ_LEN
)

nepali_train_dataset = ExampleDataset(
    nepali_train_path, nepali_tokenizer, SEQ_LEN
)

nepali_val_dataset = ExampleDataset(
    nepali_val_path, nepali_tokenizer, SEQ_LEN
)

In [ ]:
hindi_train_loader = DataLoader(
    hindi_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

hindi_val_loader = DataLoader(
    hindi_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

nepali_train_loader = DataLoader(
    nepali_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

nepali_val_loader = DataLoader(
    nepali_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

batch = next(iter(hindi_train_loader))

print(batch["input_ids"].shape)
print(batch["labels"].shape)
print("Fixed example-level DataLoaders ready.")

In [ ]:
hindi_best_val_loss = finetune_model(
    model=hindi_model,
    train_loader=hindi_train_loader,
    val_loader=hindi_val_loader,
    output_path=hindi_output_path,
    epochs=2,
    lr=5e-6,
    grad_accum_steps=4
)

In [ ]:
nepali_best_val_loss = finetune_model(
    model=nepali_model,
    train_loader=nepali_train_loader,
    val_loader=nepali_val_loader,
    output_path=nepali_output_path,
    epochs=2,
    lr=5e-6,
    grad_accum_steps=4
)

In [ ]:
hindi_ft_checkpoint = torch.load(
    hindi_output_path,
    map_location=DEVICE,
    weights_only=False
)

nepali_ft_checkpoint = torch.load(
    nepali_output_path,
    map_location=DEVICE,
    weights_only=False
)

hindi_model.load_state_dict(
    hindi_ft_checkpoint["model_state_dict"]
)

nepali_model.load_state_dict(
    nepali_ft_checkpoint["model_state_dict"]
)

hindi_model.eval()
nepali_model.eval()

print("Both best fine-tuned checkpoints loaded.")

In [ ]:
hindi_prompt = """प्रश्न: यदि रवि का मान गीता से अधिक है और गीता का मान विनय से अधिक है, तो इनमें सबसे बड़ा कौन है?

समाधान:"""

print(
    generate_greedy(
        hindi_model,
        hindi_tokenizer,
        hindi_prompt,
        max_new_tokens=50
    )
)

In [ ]:
nepali_prompt = """प्रश्न: यदि रामको अंक श्यामको भन्दा बढी छ र श्यामको अंक हरिको भन्दा बढी छ भने सबैभन्दा ठूलो अंक कसको हो?

समाधान:"""

print(
    generate_greedy(
        nepali_model,
        nepali_tokenizer,
        nepali_prompt,
        max_new_tokens=50
    )
)

In [ ]:
@torch.no_grad()
def generate_answer_only(
    model,
    tokenizer,
    prompt,
    max_new_tokens=25
):
    model.eval()

    ids = tokenizer.encode(prompt)

    input_ids = torch.tensor(
        [ids],
        dtype=torch.long,
        device=DEVICE
    )

    for _ in range(max_new_tokens):
        context = input_ids[:, -SEQ_LEN:]

        logits = model(context)

        if isinstance(logits, tuple):
            logits = logits[0]

        next_token = torch.argmax(
            logits[:, -1, :],
            dim=-1,
            keepdim=True
        )

        input_ids = torch.cat(
            [input_ids, next_token],
            dim=1
        )

    output = tokenizer.decode(
        input_ids[0].tolist()
    )

    return output

In [ ]:
hindi_demo_prompt = """प्रश्न: यदि रवि का मान गीता से अधिक है और गीता का मान विनय से अधिक है, तो इनमें सबसे बड़ा कौन है?

समाधान: रवि > गीता और गीता > विनय। इसलिए रवि सबसे बड़ा है।

उत्तर:"""

print(
    generate_answer_only(
        hindi_model,
        hindi_tokenizer,
        hindi_demo_prompt,
        max_new_tokens=10
    )
)

In [ ]:
import os
import json
import random
import torch
from torch.utils.data import Dataset, DataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEQ_LEN = 128
BATCH_SIZE = 4
NUM_WORKERS = 2

EPOCHS = 3
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01

print("Device:", DEVICE)

In [ ]:
HINDI_DATA_PATH = (
    "/kaggle/working/reasoning_project/"
    "splits/hindi/train.jsonl"
)

HINDI_VAL_PATH = (
    "/kaggle/working/reasoning_project/"
    "splits/hindi/validation.jsonl"
)

NEPALI_DATA_PATH = (
    "/kaggle/working/reasoning_project/"
    "splits/nepali/train.jsonl"
)

NEPALI_VAL_PATH = (
    "/kaggle/working/reasoning_project/"
    "splits/nepali/validation.jsonl"
)

HINDI_MODEL_PATH = (
    "/kaggle/input/datasets/bharadwajvss/"
    "trained-language-model/hindi_model.pt"
)

NEPALI_MODEL_PATH = (
    "/kaggle/input/datasets/bharadwajvss/"
    "trained-language-model/nepal_model.pt"
)

HINDI_BPE_PATH = (
    "/kaggle/input/datasets/bharadwajvss/"
    "hindi-phase2-final/hindi_bpe_31744.json"
)

NEPALI_BPE_PATH = (
    "/kaggle/input/datasets/bharadwajvss/"
    "nepali-phase2-source/nepali_bpe_31744.json"
)

In [ ]:
def load_qa_jsonl(path):
    examples = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)

            question = record.get("question", "").strip()
            answer = record.get("answer", "").strip()

            if not question or not answer:
                continue

            text = (
                f"प्रश्न: {question}\n\n"
                f"उत्तर: {answer}"
            )

            examples.append({
                "question": question,
                "answer": answer,
                "text": text
            })

    return examples

In [ ]:
hindi_train_examples = load_qa_jsonl(HINDI_DATA_PATH)
hindi_val_examples = load_qa_jsonl(HINDI_VAL_PATH)

nepali_train_examples = load_qa_jsonl(NEPALI_DATA_PATH)
nepali_val_examples = load_qa_jsonl(NEPALI_VAL_PATH)

print("Hindi train:", len(hindi_train_examples))
print("Hindi validation:", len(hindi_val_examples))

print("Nepali train:", len(nepali_train_examples))
print("Nepali validation:", len(nepali_val_examples))

print("\nExample:")
print(hindi_train_examples[0]["text"])

In [ ]:
class QAExampleDataset(Dataset):
    def __init__(self, examples, tokenizer, seq_len=128):
        self.examples = examples
        self.tokenizer = tokenizer
        self.seq_len = seq_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        text = self.examples[idx]["text"]

        token_ids = self.tokenizer.encode(text)

        # Add one extra token because input and target are shifted.
        token_ids = token_ids[:self.seq_len + 1]

        input_ids = token_ids[:-1]
        labels = token_ids[1:]

        # Pad short examples.
        pad_length = self.seq_len - len(input_ids)

        if pad_length > 0:
            input_ids = input_ids + [0] * pad_length
            labels = labels + [-100] * pad_length

        return {
            "input_ids": torch.tensor(
                input_ids,
                dtype=torch.long
            ),
            "labels": torch.tensor(
                labels,
                dtype=torch.long
            )
        }

In [ ]:
from torch.utils.data import DataLoader

def create_qa_loader(
    examples,
    tokenizer,
    batch_size=16,
    shuffle=True
):
    dataset = QAExampleDataset(
        examples,
        tokenizer
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle
    )

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()

        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.d_model = d_model

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim
        ).transpose(1, 2)

        # Causal attention: tokens cannot see future tokens
        attention_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device),
            diagonal=1
        ).bool()

        attention_scores = attention_scores.masked_fill(
            causal_mask,
            float("-inf")
        )

        attention_weights = F.softmax(attention_scores, dim=-1)
        attention_weights = self.dropout(attention_weights)

        output = attention_weights @ v

        output = output.transpose(1, 2).contiguous()
        output = output.view(batch_size, seq_len, d_model)

        return self.out_proj(output)


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)

        self.attn = CausalSelfAttention(
            d_model=d_model,
            num_heads=num_heads,
            dropout=dropout
        )

        self.ln2 = nn.LayerNorm(d_model)

        # Important: checkpoint expects ff.net.0 and ff.net.3
        self.ff = FeedForward(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.attn(self.ln1(x)))
        x = x + self.dropout(self.ff(self.ln2(x)))
        return x

class ReasoningTransformer(nn.Module):
    def __init__(self, config, final_norm_name="final_norm"):
        super().__init__()

        self.vocab_size = config["vocab_size"]
        self.d_model = config["d_model"]
        self.seq_len = config["seq_len"]

        self.token_embedding = nn.Embedding(
            self.vocab_size,
            self.d_model
        )

        self.position_embedding = nn.Embedding(
            self.seq_len,
            self.d_model
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(
                d_model=config["d_model"],
                num_heads=config["num_heads"],
                d_ff=config["d_ff"],
                dropout=config["dropout"]
            )
            for _ in range(config["num_layers"])
        ])

        # The checkpoint uses different names for Hindi and Nepali.
        if final_norm_name == "final_norm":
            self.final_norm = nn.LayerNorm(self.d_model)
        else:
            self.ln_f = nn.LayerNorm(self.d_model)

        self.lm_head = nn.Linear(
            self.d_model,
            self.vocab_size
        )

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape

        if seq_len > self.seq_len:
            raise ValueError(
                f"Sequence length {seq_len} exceeds maximum {self.seq_len}"
            )

        positions = torch.arange(
            seq_len,
            device=input_ids.device
        )

        x = self.token_embedding(input_ids)
        x = x + self.position_embedding(positions)

        for block in self.blocks:
            x = block(x)

        if hasattr(self, "final_norm"):
            x = self.final_norm(x)
        else:
            x = self.ln_f(x)

        logits = self.lm_head(x)

        return logits

In [ ]:
def load_model(checkpoint_path):
    config = {
        "vocab_size": 31744,
        "d_model": 288,
        "num_heads": 8,
        "num_layers": 6,
        "d_ff": 1440,
        "seq_len": 128,
        "dropout": 0.0,
    }

    model = ReasoningTransformer(config).to(DEVICE)

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE
    )

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint

    model.load_state_dict(state_dict)
    model.train()

    return model

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0.0
    total_batches = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad()

        logits = model(input_ids)

        if isinstance(logits, tuple):
            logits = logits[0]

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

    return total_loss / max(total_batches, 1)

In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_batches = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        logits = model(input_ids)

        if isinstance(logits, tuple):
            logits = logits[0]

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            labels.reshape(-1)
        )

        total_loss += loss.item()
        total_batches += 1

    return total_loss / max(total_batches, 1)

In [ ]:
import json


class RegexBPETokenizer:
    def __init__(self, tokenizer_data):
        self.format = tokenizer_data["format"]
        self.language = tokenizer_data["language"]
        self.vocab_size = tokenizer_data["vocab_size"]
        self.base_vocab_size = tokenizer_data["base_vocab_size"]
        self.pattern = tokenizer_data["pattern"]
        self.special_tokens = tokenizer_data["special_tokens"]
        self.merges = tokenizer_data["merges"]

        # Every merge is:
        # [left_token_id, right_token_id, new_token_id]
        self.merge_ranks = {}

        for rank, merge in enumerate(self.merges):
            left_id, right_id, new_id = merge

            self.merge_ranks[(left_id, right_id)] = (
                rank,
                new_id
            )

        # Token ID -> raw bytes.
        self.id_to_bytes = {
            token_id: bytes([token_id])
            for token_id in range(self.base_vocab_size)
        }

        # Reconstruct the byte representation of every merged token.
        for merge in self.merges:
            left_id, right_id, new_id = merge

            if left_id not in self.id_to_bytes:
                raise ValueError(
                    f"Missing left token ID: {left_id}"
                )

            if right_id not in self.id_to_bytes:
                raise ValueError(
                    f"Missing right token ID: {right_id}"
                )

            self.id_to_bytes[new_id] = (
                self.id_to_bytes[left_id]
                + self.id_to_bytes[right_id]
            )

    @classmethod
    def from_file(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            tokenizer_data = json.load(f)

        return cls(tokenizer_data)

    def encode(self, text):
        # Start with UTF-8 byte IDs.
        tokens = list(text.encode("utf-8"))

        if not tokens:
            return []

        # Apply the highest-priority available merge repeatedly.
        while len(tokens) >= 2:
            best_rank = None
            best_position = None
            best_new_id = None

            for i in range(len(tokens) - 1):
                pair = (
                    tokens[i],
                    tokens[i + 1]
                )

                if pair not in self.merge_ranks:
                    continue

                rank, new_id = self.merge_ranks[pair]

                if best_rank is None or rank < best_rank:
                    best_rank = rank
                    best_position = i
                    best_new_id = new_id

            if best_position is None:
                break

            i = best_position

            tokens = (
                tokens[:i]
                + [best_new_id]
                + tokens[i + 2:]
            )

        return tokens

    def decode(self, token_ids):
        byte_data = bytearray()

        for token_id in token_ids:
            token_id = int(token_id)

            if token_id not in self.id_to_bytes:
                raise ValueError(
                    f"Unknown token ID: {token_id}"
                )

            byte_data.extend(
                self.id_to_bytes[token_id]
            )

        return bytes(byte_data).decode(
            "utf-8",
            errors="replace"
        )

In [ ]:
tokenizer = RegexBPETokenizer.from_file(HINDI_BPE_PATH)

hindi_train_loader = create_qa_loader(
    hindi_train_examples,
    tokenizer,
    shuffle=True
)

hindi_val_loader = create_qa_loader(
    hindi_val_examples,
    tokenizer,
    shuffle=False
)

model = load_model(HINDI_MODEL_PATH)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

criterion = torch.nn.CrossEntropyLoss(
    ignore_index=-100
)

best_val_loss = float("inf")

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(
        model,
        hindi_train_loader,
        optimizer,
        criterion
    )

    val_loss = evaluate(
        model,
        hindi_val_loader,
        criterion
    )

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "config": {
                    "vocab_size": 31744,
                    "d_model": 288,
                    "num_heads": 8,
                    "num_layers": 6,
                    "d_ff": 1440,
                    "seq_len": 128
                },
                "best_val_loss": best_val_loss
            },
            "/kaggle/working/reasoning_project/"
            "hindi_qa_finetuned.pt"
        )

        print("Saved best Hindi checkpoint.")

In [ ]:
HINDI_QA_MODEL_PATH = (
    "/kaggle/working/reasoning_project/"
    "hindi_qa_finetuned.pt"
)

model = load_model(HINDI_QA_MODEL_PATH)
tokenizer = RegexBPETokenizer.from_file(HINDI_BPE_PATH)

print("Hindi QA model loaded.")

In [ ]:
import re
import torch


def generate_greedy(
    model,
    tokenizer,
    prompt,
    max_new_tokens=40
):
    model.eval()

    input_ids = torch.tensor(
        [tokenizer.encode(prompt)],
        dtype=torch.long,
        device=DEVICE
    )

    generated = input_ids.clone()

    with torch.no_grad():
        for _ in range(max_new_tokens):
            context = generated[:, -SEQ_LEN:]

            logits = model(context)

            if isinstance(logits, tuple):
                logits = logits[0]

            next_token = torch.argmax(
                logits[:, -1, :],
                dim=-1,
                keepdim=True
            )

            generated = torch.cat(
                [generated, next_token],
                dim=1
            )

            decoded = tokenizer.decode(
                generated[0].tolist()
            )

            # Stop when a complete answer line is generated.
            answer_match = re.search(
                r"उत्तर\s*:\s*([^\n]+)",
                decoded
            )

            if answer_match:
                answer = answer_match.group(1).strip()

                if answer:
                    break

    return tokenizer.decode(
        generated[0].tolist()
    )

In [ ]:
prompt = """प्रश्न: यदि रवि का मान गीता से अधिक है और गीता का मान विनय से अधिक है, तो इनमें सबसे बड़ा कौन है?

उत्तर:"""

raw_output = generate_greedy(
    model,
    tokenizer,
    prompt,
    max_new_tokens=40
)

print(raw_output)

In [ ]:
test_questions = [
    "यदि रवि, गीता से बड़ा है और गीता, विनय से बड़ी है, तो सबसे बड़ा कौन है?",
    "यदि राम, श्याम से बड़ा है और श्याम, मोहन से बड़ा है, तो सबसे बड़ा कौन है?",
    "यदि अमित, भारत से बड़ा है और भारत, चेतन से बड़ा है, तो सबसे बड़ा कौन है?",
    "यदि सोहन, रोहन से बड़ा है और रोहन, मोहन से बड़ा है, तो सबसे बड़ा कौन है?",
    "यदि सीता के पास गीता से अधिक पुस्तकें हैं, तो किसके पास अधिक पुस्तकें हैं?"
]

for question in test_questions:
    prompt = f"प्रश्न: {question}\n\nउत्तर:"

    output = generate_greedy(
        model,
        tokenizer,
        prompt,
        max_new_tokens=20
    )

    print("=" * 70)
    print(output)

In [ ]:
symbolic_questions = [
    "यदि A, B से बड़ा है और B, C से बड़ा है, तो सबसे बड़ा कौन है?",
    "यदि X, Y से बड़ा है और Y, Z से बड़ा है, तो सबसे बड़ा कौन है?",
    "यदि P, Q से छोटा है और Q, R से छोटा है, तो सबसे छोटा कौन है?"
]

for question in symbolic_questions:
    prompt = f"प्रश्न: {question}\n\nउत्तर:"

    output = generate_greedy(
        model,
        tokenizer,
        prompt,
        max_new_tokens=20
    )

    print("=" * 70)
    print(output)

In [ ]:
for i in range(20):
    print(hindi_train_examples[i]["text"])
    print("-" * 60)

In [ ]:
from collections import Counter

answer_counts = Counter(
    example["answer"]
    for example in hindi_train_examples
)

print(answer_counts.most_common(30))

In [ ]:
comparison_examples = [
    example
    for example in hindi_train_examples
    if any(
        word in example["question"]
        for word in [
            "सबसे बड़ा",
            "सबसे बड़ी",
            "सबसे छोटा",
            "सबसे छोटी",
            "अधिक",
            "कम"
        ]
    )
]

print("Comparison examples:", len(comparison_examples))

for example in comparison_examples[:20]:
    print("QUESTION:", example["question"])
    print("ANSWER:", example["answer"])
    print("-" * 60)

In [ ]:
test_questions = [
    "यदि रवि का मान गीता से अधिक है और गीता का मान विनय से अधिक है, तो इनमें सबसे बड़ा कौन है?",
    "यदि राम का मान श्याम से अधिक है और श्याम का मान मोहन से अधिक है, तो इनमें सबसे बड़ा कौन है?",
    "यदि अमित का मान भारत से अधिक है और भारत का मान चेतन से अधिक है, तो इनमें सबसे बड़ा कौन है?",
    "यदि सोहन का मान रोहन से अधिक है और रोहन का मान मोहन से अधिक है, तो इनमें सबसे बड़ा कौन है?",
    "यदि सीता के पास गीता से अधिक पुस्तकें हैं, तो किसके पास अधिक पुस्तकें हैं?"
]

for question in test_questions:
    prompt = f"प्रश्न: {question}\n\nउत्तर:"

    output = generate_greedy(
        model,
        tokenizer,
        prompt,
        max_new_tokens=15
    )

    print("=" * 70)
    print(output)

In [ ]:
comparison_examples = [
    (i, example)
    for i, example in enumerate(hindi_train_examples)
    if any(
        phrase in example["question"]
        for phrase in [
            "सबसे बड़ा",
            "सबसे बड़ी",
            "सबसे छोटा",
            "सबसे छोटी",
        ]
    )
]

print("Comparison examples:", len(comparison_examples))

for idx, example in comparison_examples[:20]:
    print("INDEX:", idx)
    print("QUESTION:", example["question"])
    print("ANSWER:", example["answer"])
    print("-" * 80)

In [ ]:
for idx, example in comparison_examples[:5]:
    question = example["question"]
    expected = example["answer"]

    generated = generate_greedy(
        model,
        tokenizer,
        question,
        max_new_tokens=20
    )

    print("QUESTION:", question)
    print("EXPECTED:", expected)
    print("GENERATED:", generated)
    print("=" * 100)

In [ ]:
for idx, example in comparison_examples[:5]:
    question = example["question"]
    expected = example["answer"]

    prompt = f"प्रश्न: {question}\n\nउत्तर:"

    generated = generate_greedy(
        model,
        tokenizer,
        prompt,
        max_new_tokens=10
    )

    print("QUESTION:", question)
    print("EXPECTED:", expected)
    print("GENERATED:", generated)
    print("=" * 100)

In [ ]:
def answer_hindi_question(model, tokenizer, question, max_new_tokens=15):
    prompt = f"प्रश्न: {question}\n\nउत्तर:"

    generated = generate_greedy(
        model,
        tokenizer,
        prompt,
        max_new_tokens=max_new_tokens
    )

    if "उत्तर:" in generated:
        answer = generated.split("उत्तर:", 1)[1]
    else:
        answer = generated

    # Stop at possible next sections
    answer = answer.split("\n")[0].strip()

    return answer

In [ ]:
test_questions = [
    "यदि विनय का मान पूजा से अधिक है और पूजा का मान अजय से अधिक है, तो इनमें सबसे छोटा कौन है?",
    "यदि नेहा का मान सीमा से अधिक है और सीमा का मान अनिल से अधिक है, तो इनमें सबसे छोटा कौन है?",
    "यदि पूजा का मान मीना से अधिक है और मीना का मान सोनम से अधिक है, तो इनमें सबसे बड़ा कौन है?",
    "चार व्यक्तियों के मान इस प्रकार हैं: मोहन > अमित, अमित > विनय, और विनय > सीता। इनमें सबसे छोटा कौन है?",
]

for question in test_questions:
    print("QUESTION:", question)
    print("ANSWER:", answer_hindi_question(model, tokenizer, question))
    print("-" * 100)

In [ ]:
natural_questions = [
    "यदि रवि, गीता से बड़ा है और गीता, विनय से बड़ी है, तो सबसे बड़ा कौन है?",
    "यदि राम, श्याम से बड़ा है और श्याम, मोहन से बड़ा है, तो सबसे छोटा कौन है?",
    "यदि अमित, पूजा से बड़ा है और पूजा, सीमा से बड़ी है, तो सबसे बड़ा कौन है?",
]

for question in natural_questions:
    print("QUESTION:", question)
    print("ANSWER:", answer_hindi_question(model, tokenizer, question))
    print("-" * 100)

In [ ]:
natural_comparison_examples = [
    {
        "question": "यदि रवि, गीता से बड़ा है और गीता, विनय से बड़ी है, तो सबसे बड़ा कौन है?",
        "answer": "रवि"
    },
    {
        "question": "यदि राम, श्याम से बड़ा है और श्याम, मोहन से बड़ा है, तो सबसे छोटा कौन है?",
        "answer": "मोहन"
    },
    {
        "question": "यदि अमित, पूजा से बड़ा है और पूजा, सीमा से बड़ी है, तो सबसे बड़ा कौन है?",
        "answer": "अमित"
    },
    {
        "question": "यदि सीता, राधा से छोटी है और राधा, गीता से छोटी है, तो सबसे छोटी कौन है?",
        "answer": "सीता"
    },
    {
        "question": "यदि मोहन, अमित से छोटा है और अमित, विनय से छोटा है, तो सबसे बड़ा कौन है?",
        "answer": "विनय"
    },
    {
        "question": "यदि नेहा, सीमा से लंबी है और सीमा, अनिल से लंबी है, तो सबसे छोटा कौन है?",
        "answer": "अनिल"
    },
    {
        "question": "यदि पूजा, मीना से अधिक है और मीना, सोनम से अधिक है, तो सबसे बड़ी कौन है?",
        "answer": "पूजा"
    },
    {
        "question": "यदि विकास, राम से कम है और राम, सुरेश से कम है, तो सबसे अधिक किसका मान है?",
        "answer": "सुरेश"
    },
    {
        "question": "यदि गीता, कविता से बड़ी है और कविता, प्रिया से बड़ी है, तो सबसे छोटी कौन है?",
        "answer": "प्रिया"
    },
    {
        "question": "यदि अजय, विनय से छोटा है और विनय, रोहित से छोटा है, तो सबसे बड़ा कौन है?",
        "answer": "रोहित"
    },
]

In [ ]:
augmented_train_examples = [
    {
        "text": f"प्रश्न: {example['question']}\n\nउत्तर: {example['answer']}"
    }
    for example in augmented_train_examples
]

print("Converted examples:", len(augmented_train_examples))
print(augmented_train_examples[0])

In [ ]:
augmented_loader = create_qa_loader(
    augmented_train_examples,
    tokenizer,
    batch_size=16,
    shuffle=True
)

print("Loader recreated successfully")
print("Number of batches:", len(augmented_loader))

In [ ]:
batch = next(iter(augmented_loader))

print(type(batch))

if isinstance(batch, dict):
    for key, value in batch.items():
        print(key, value.shape)
else:
    print(len(batch))
    for item in batch:
        print(item.shape)

In [ ]:
model = load_model(
    "/kaggle/working/reasoning_project/hindi_qa_finetuned.pt"
)

model = model.to(device)
model.train()

print("Model loaded successfully")

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=5e-6,
    weight_decay=0.01
)

print("Optimizer created successfully")

In [ ]:
criterion = torch.nn.CrossEntropyLoss(
    ignore_index=-100
)

print("Loss function created successfully")

In [ ]:
train_loss = train_one_epoch(
    model,
    augmented_loader,
    optimizer,
    criterion
)

print(f"Training loss: {train_loss:.4f}")

In [ ]:
checkpoint_path = (
    "/kaggle/working/reasoning_project/"
    "hindi_qa_natural_comparison_finetuned.pt"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": {
            "vocab_size": 31744,
            "d_model": 288,
            "num_heads": 8,
            "num_layers": 6,
            "d_ff": 1440,
            "seq_len": 128
        }
    },
    checkpoint_path
)

print("Saved:", checkpoint_path)

In [ ]:
model.eval()

natural_questions = [
    "यदि रवि, गीता से बड़ा है और गीता, विनय से बड़ी है, तो सबसे बड़ा कौन है?",
    "यदि राम, श्याम से बड़ा है और श्याम, मोहन से बड़ा है, तो सबसे छोटा कौन है?",
    "यदि अमित, पूजा से बड़ा है और पूजा, सीमा से बड़ी है, तो सबसे बड़ा कौन है?",
]

for question in natural_questions:
    answer = answer_hindi_question(
        model,
        tokenizer,
        question,
        max_new_tokens=15
    )

    print("QUESTION:", question)
    print("ANSWER:", answer)
    print("-" * 100)

In [ ]:
model.eval()

all_test_questions = [
    (
        "यदि विनय का मान पूजा से अधिक है और पूजा का मान अजय से अधिक है, "
        "तो इनमें सबसे छोटा कौन है?",
        "अजय"
    ),
    (
        "यदि पूजा का मान मीना से अधिक है और मीना का मान सोनम से अधिक है, "
        "तो इनमें सबसे बड़ा कौन है?",
        "पूजा"
    ),
    (
        "यदि रवि, गीता से बड़ा है और गीता, विनय से बड़ी है, "
        "तो सबसे बड़ा कौन है?",
        "रवि"
    ),
    (
        "यदि राम, श्याम से बड़ा है और श्याम, मोहन से बड़ा है, "
        "तो सबसे छोटा कौन है?",
        "मोहन"
    ),
]

correct = 0

for question, expected in all_test_questions:
    predicted = answer_hindi_question(
        model,
        tokenizer,
        question,
        max_new_tokens=15
    )

    is_correct = predicted == expected
    correct += int(is_correct)

    print("QUESTION:", question)
    print("EXPECTED:", expected)
    print("PREDICTED:", predicted)
    print("CORRECT:", is_correct)
    print("-" * 100)

print(f"Accuracy: {correct}/{len(all_test_questions)}")

In [ ]:
final_checkpoint_path = (
    "/kaggle/working/reasoning_project/"
    "hindi_qa_natural_comparison_final.pt"
)

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "config": {
            "vocab_size": 31744,
            "d_model": 288,
            "num_heads": 8,
            "num_layers": 6,
            "d_ff": 1440,
            "seq_len": 128
        },
        "training_info": {
            "task": "Hindi QA with natural comparison augmentation",
            "comparison_test_accuracy": "4/4"
        }
    },
    final_checkpoint_path
)

print("Final checkpoint saved:")
print(final_checkpoint_path)

In [ ]:
final_model = load_model(
    "/kaggle/working/reasoning_project/"
    "hindi_qa_natural_comparison_final.pt"
)

final_model = final_model.to(device)
final_model.eval()

print("Final checkpoint loaded successfully")

In [ ]:
test_questions = [
    "यदि विनय का मान पूजा से अधिक है और पूजा का मान अजय से अधिक है, तो इनमें सबसे छोटा कौन है?",
    "यदि पूजा का मान मीना से अधिक है और मीना का मान सोनम से अधिक है, तो इनमें सबसे बड़ा कौन है?",
    "यदि रवि, गीता से बड़ा है और गीता, विनय से बड़ी है, तो सबसे बड़ा कौन है?",
    "यदि राम, श्याम से बड़ा है और श्याम, मोहन से बड़ा है, तो सबसे छोटा कौन है?",
]

for question in test_questions:
    answer = answer_hindi_question(
        final_model,
        tokenizer,
        question,
        max_new_tokens=15
    )

    print("QUESTION:", question)
    print("ANSWER:", answer)
    print("-" * 100)

In [ ]:
import os

checkpoint_path = (
    "/kaggle/working/reasoning_project/"
    "hindi_qa_natural_comparison_final.pt"
)

print("Exists:", os.path.exists(checkpoint_path))

if os.path.exists(checkpoint_path):
    print("File size:", round(os.path.getsize(checkpoint_path) / (1024 ** 2), 2), "MB")
    print("Checkpoint:", checkpoint_path)

In [8]:
nepali_tokenizer_path = (
    "/kaggle/input/datasets/bharadwajvss/"
    "nepali-phase2-source/nepali_bpe_31744.json"
)

nepali_tokenizer = RegexBPE(
    nepali_tokenizer_path
)

print("Nepali tokenizer loaded successfully")
print("Vocab size:", nepali_tokenizer.vocab_size)

NameError: name 'RegexBPE' is not defined

In [ ]:
nepali_train_path = (
    "/kaggle/working/reasoning_project/"
    "splits/nepali/train.jsonl"
)

nepali_train_examples = load_qa_jsonl(
    nepali_train_path
)

print("Nepali training examples:", len(nepali_train_examples))
print(nepali_train_examples[0])

In [ ]:
nepali_loader = create_qa_loader(
    nepali_train_examples,
    nepali_tokenizer,
    batch_size=16,
    shuffle=True
)

print("Nepali loader created successfully")
print("Number of batches:", len(nepali_loader))

In [ ]:
import inspect

print(inspect.signature(ReasoningTransformer))
print(inspect.signature(ReasoningTransformer.__init__))

In [7]:
nepali_config = {
    "vocab_size": 31744,
    "d_model": 288,
    "num_heads": 8,
    "num_layers": 6,
    "d_ff": 1440,
    "seq_len": 128,
    "dropout": 0.0
}

nepali_model = ReasoningTransformer(
    nepali_config,
    final_norm_name="final_norm"
)

nepali_model.load_state_dict(state_dict)

nepali_model = nepali_model.to(device)
nepali_model.train()

print("Nepali model loaded successfully")

TypeError: ReasoningTransformer.__init__() got an unexpected keyword argument 'final_norm_name'

In [ ]:
import torch

def generate_text(model, tokenizer, prompt, max_new_tokens=80, temperature=0.8, top_k=50):
    model.eval()

    input_ids = tokenizer.encode(prompt)
    input_ids = torch.tensor([input_ids], dtype=torch.long, device=device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            input_cond = input_ids[:, -nepali_config["seq_len"]:]

            logits = model(input_cond)
            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                values, indices = torch.topk(logits, top_k)
                filtered_logits = torch.full_like(logits, float("-inf"))
                filtered_logits.scatter_(1, indices, values)
                logits = filtered_logits

            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            input_ids = torch.cat([input_ids, next_token], dim=1)

    return tokenizer.decode(input_ids[0].tolist())


prompt = "नेपाल एक सुन्दर"
output = generate_text(nepali_model, nepali_tokenizer, prompt)

print(output)

In [ ]:
prompts = [
    "नेपालको राजधानी",
    "नेपालको प्राकृतिक सुन्दरता",
    "कृषि क्षेत्रको विकास",
    "शिक्षाको महत्व",
    "आजको मौसम"
]

for prompt in prompts:
    print(f"\nPrompt: {prompt}")
    print(generate_text(
        nepali_model,
        nepali_tokenizer,
        prompt,
        max_new_tokens=60,
        temperature=0.7,
        top_k=50
    ))

In [ ]:
import math
import torch

nepali_model.eval()

total_loss = 0.0
total_tokens = 0

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        targets = batch["targets"].to(device)

        logits = nepali_model(input_ids)

        loss = torch.nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1),
            ignore_index=-100,
            reduction="sum"
        )

        total_loss += loss.item()
        total_tokens += (targets != -100).sum().item()

val_loss = total_loss / total_tokens
perplexity = math.exp(val_loss)

print(f"Validation Loss: {val_loss:.4f}")
print(f"Perplexity: {perplexity:.2f}")

In [ ]:
nepali_test_path = "/kaggle/working/reasoning_project/splits/nepali/test.jsonl"

nepali_test_dataset = NepaliDataset(
    nepali_test_path,
    tokenizer=nepali_tokenizer,
    seq_len=nepali_config["seq_len"]
)

test_loader = DataLoader(
    nepali_test_dataset,
    batch_size=8,
    shuffle=False
)

print("Test loader created:", len(test_loader))

In [ ]:
import os

print("Kaggle input datasets:\n")

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files[:20]:
        print(f"{indent}  {file}")

In [ ]:
import os
import shutil

# Working directories
BASE_DIR = "/kaggle/working/reasoning_project"
NEPALI_DIR = os.path.join(BASE_DIR, "nepali")
HINDI_DIR = os.path.join(BASE_DIR, "hindi")

os.makedirs(NEPALI_DIR, exist_ok=True)
os.makedirs(HINDI_DIR, exist_ok=True)

# Input locations
NEPALI_SOURCE = "/kaggle/input/datasets/bharadwajvss/nepali-phase2-source"
MODEL_SOURCE = "/kaggle/input/datasets/bharadwajvss/trained-language-model"
HINDI_SOURCE = "/kaggle/input/datasets/bharadwajvss/hindi-phase2-final"

# Copy Nepali QA data and tokenizer
shutil.copy(
    os.path.join(MODEL_SOURCE, "nepali_reasoning_deduplicated.jsonl"),
    os.path.join(NEPALI_DIR, "nepali_reasoning_deduplicated.jsonl")
)

shutil.copy(
    os.path.join(NEPALI_SOURCE, "nepali_bpe_31744.json"),
    os.path.join(NEPALI_DIR, "nepali_bpe_31744.json")
)

# Copy original Nepali model
shutil.copy(
    os.path.join(MODEL_SOURCE, "nepal_model.pt"),
    os.path.join(NEPALI_DIR, "nepal_model.pt")
)

# Copy Hindi fine-tuning data and model
for filename in [
    "hindi_train_00000.jsonl",
    "hindi_validation_00000.jsonl",
    "hindi_test_00000.jsonl",
    "hindi_bpe_31744.json"
]:
    shutil.copy(
        os.path.join(HINDI_SOURCE, filename),
        os.path.join(HINDI_DIR, filename)
    )

shutil.copy(
    os.path.join(MODEL_SOURCE, "hindi_model.pt"),
    os.path.join(HINDI_DIR, "hindi_model.pt")
)

print("Working directory recreated successfully.")
print("Nepali files:", os.listdir(NEPALI_DIR))
print("Hindi files:", os.listdir(HINDI_DIR))

In [ ]:
import json
import torch

nepali_qa_path = "/kaggle/working/reasoning_project/nepali/nepali_reasoning_deduplicated.jsonl"
nepali_model_path = "/kaggle/working/reasoning_project/nepali/nepal_model.pt"

with open(nepali_qa_path, "r", encoding="utf-8") as f:
    for i in range(3):
        item = json.loads(next(f))
        print(f"\n--- QA SAMPLE {i + 1} ---")
        print(json.dumps(item, ensure_ascii=False, indent=2))

checkpoint = torch.load(nepali_model_path, map_location="cpu")

print("\n--- CHECKPOINT INFO ---")
print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:", list(checkpoint.keys())[:30])

    state_dict = checkpoint.get("model_state_dict", checkpoint)

    print("Number of parameters:", len(state_dict))
    print("First parameter names:")
    for name in list(state_dict.keys())[:15]:
        print(name)
else:
    state_dict = checkpoint
    print("Number of parameters:", len(state_dict))

In [5]:
checkpoint = torch.load(
    "/kaggle/working/reasoning_project/nepali/nepal_model.pt",
    map_location="cpu"
)

print("Original Nepali training step:", checkpoint.get("step"))
print("Tokens seen:", checkpoint.get("tokens_seen"))
print("\nOriginal model config:")
print(checkpoint["config"])

Original Nepali training step: 61035
Tokens seen: 499998720

Original model config:
{'vocab_size': 31744, 'd_model': 288, 'num_heads': 8, 'num_layers': 6, 'd_ff': 1440, 'seq_len': 128, 'dropout': 0.1, 'batch_size': 16, 'grad_accum_steps': 4, 'lr': 0.0003, 'weight_decay': 0.1, 'max_grad_norm': 1.0, 'target_tokens': 500000000, 'warmup_steps': 2000}


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        d_model = config["d_model"]
        num_heads = config["num_heads"]

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(config["dropout"])

        mask = torch.tril(
            torch.ones(config["seq_len"], config["seq_len"])
        ).view(1, 1, config["seq_len"], config["seq_len"])

        # Do not save this buffer in the checkpoint
        self.register_buffer("mask", mask, persistent=False)

    def forward(self, x):
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        att = att.masked_fill(
            self.mask[:, :, :T, :T] == 0,
            float("-inf")
        )

        att = F.softmax(att, dim=-1)
        att = self.dropout(att)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)

        return self.out_proj(y)


class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(config["d_model"], config["d_ff"]),
            nn.GELU(),
            nn.Dropout(config["dropout"]),
            nn.Linear(config["d_ff"], config["d_model"])
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.ln1 = nn.LayerNorm(config["d_model"])
        self.attn = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config["d_model"])
        self.ff = FeedForward(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class ReasoningTransformer(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.token_embedding = nn.Embedding(
            config["vocab_size"],
            config["d_model"]
        )

        self.position_embedding = nn.Embedding(
            config["seq_len"],
            config["d_model"]
        )

        self.blocks = nn.ModuleList([
            TransformerBlock(config)
            for _ in range(config["num_layers"])
        ])

        self.ln_f = nn.LayerNorm(config["d_model"])

        self.lm_head = nn.Linear(
            config["d_model"],
            config["vocab_size"],
            bias=True
        )

    def forward(self, idx):
        B, T = idx.shape

        positions = torch.arange(
            T,
            device=idx.device
        ).unsqueeze(0)

        x = self.token_embedding(idx) + self.position_embedding(positions)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)

        return self.lm_head(x)


nepali_model = ReasoningTransformer(nepali_config)

nepali_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

nepali_model = nepali_model.to(device)

print("Nepali base model loaded successfully.")

NameError: name 'nepali_config' is not defined

In [2]:
import json
import random
from torch.utils.data import Dataset, DataLoader

nepali_tokenizer_path = (
    "/kaggle/working/reasoning_project/nepali/nepali_bpe_31744.json"
)

nepali_qa_path = (
    "/kaggle/working/reasoning_project/nepali/"
    "nepali_reasoning_deduplicated.jsonl"
)

with open(nepali_qa_path, "r", encoding="utf-8") as f:
    nepali_records = [json.loads(line) for line in f]

random.seed(42)
random.shuffle(nepali_records)

n = len(nepali_records)
train_end = int(0.90 * n)
val_end = int(0.95 * n)

nepali_train_records = nepali_records[:train_end]
nepali_val_records = nepali_records[train_end:val_end]
nepali_test_records = nepali_records[val_end:]

print("Total records:", n)
print("Train records:", len(nepali_train_records))
print("Validation records:", len(nepali_val_records))
print("Test records:", len(nepali_test_records))

Total records: 9681
Train records: 8712
Validation records: 484
Test records: 485


In [3]:
tokenizer_path = (
    "/kaggle/working/reasoning_project/nepali/nepali_bpe_31744.json"
)

with open(tokenizer_path, "r", encoding="utf-8") as f:
    print(f.read(1000))

{
  "format": "regex_bpe_v1",
  "language": "nepali",
  "vocab_size": 31744,
  "base_vocab_size": 256,
  "pattern": "\\s+|\\p{L}[\\p{M}\\p{L}]*|\\p{N}+|[^\\p{L}\\p{N}\\s]",
  "special_tokens": {},
  "merges": [
    [
      224,
      164,
      256
    ],
    [
      224,
      165,
      257
    ],
    [
      256,
      190,
      258
    ],
    [
      258,
      256,
      259
    ],
    [
      257,
      141,
      260
    ],
    [
      260,
      256,
      261
    ],
    [
      256,
      149,
      262
    ],
    [
      256,
      168,
      263
    ],
    [
      256,
      176,
      264
    ],
    [
      256,
      191,
      265
    ],
    [
      257,
      135,
      266
    ],
    [
      257,
      139,
      267
    ],
    [
      256,
      184,
      268
    ],
    [
      256,
      174,
      269
    ],
    [
      256,
      170,
      270
    ],
    [
      256,
      178,
      271
    ],
    [
      256,
      164,
      272
    ],
    [
      164,
      1

In [4]:
import json
import re
import torch
from torch.utils.data import Dataset, DataLoader


class RegexBPETokenizer:
    def __init__(self, path):
        with open(path, "r", encoding="utf-8") as f:
            config = json.load(f)

        self.vocab_size = config["vocab_size"]
        self.pattern = re.compile(r"\s+|[^\W\d_][\w]*|[0-9]+|[^\w\s]", re.UNICODE)

        self.merges = {
            (int(a), int(b)): int(new_id)
            for a, b, new_id in config["merges"]
        }

        self.decode_map = {
            i: bytes([i])
            for i in range(256)
        }

        for (a, b), new_id in self.merges.items():
            self.decode_map[new_id] = (
                self.decode_map[a] + self.decode_map[b]
            )

    def encode(self, text):
        token_ids = []

        for piece in re.findall(self.pattern, text):
            ids = list(piece.encode("utf-8"))

            while len(ids) >= 2:
                candidates = [
                    (self.merges[(ids[i], ids[i + 1])], i)
                    for i in range(len(ids) - 1)
                    if (ids[i], ids[i + 1]) in self.merges
                ]

                if not candidates:
                    break

                _, index = min(candidates)

                a, b = ids[index], ids[index + 1]
                ids[index:index + 2] = [self.merges[(a, b)]]

            token_ids.extend(ids)

        return token_ids

    def decode(self, token_ids):
        byte_data = b"".join(
            self.decode_map[token_id]
            for token_id in token_ids
            if token_id in self.decode_map
        )

        return byte_data.decode("utf-8", errors="replace")


nepali_tokenizer = RegexBPETokenizer(
    "/kaggle/working/reasoning_project/nepali/nepali_bpe_31744.json"
)


class NepaliReasoningDataset(Dataset):
    def __init__(self, records, tokenizer, seq_len):
        self.samples = []
        self.seq_len = seq_len

        for item in records:
            token_ids = tokenizer.encode(item["text"])
            token_ids = token_ids[:seq_len + 1]

            if len(token_ids) >= 2:
                self.samples.append(token_ids)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        tokens = self.samples[idx]

        input_ids = tokens[:-1]
        targets = tokens[1:]

        pad_length = self.seq_len - len(input_ids)

        input_ids += [0] * pad_length
        targets += [-100] * pad_length

        return {
            "input_ids": torch.tensor(
                input_ids[:self.seq_len],
                dtype=torch.long
            ),
            "targets": torch.tensor(
                targets[:self.seq_len],
                dtype=torch.long
            )
        }


nepali_train_dataset = NepaliReasoningDataset(
    nepali_train_records,
    nepali_tokenizer,
    nepali_config["seq_len"]
)

nepali_val_dataset = NepaliReasoningDataset(
    nepali_val_records,
    nepali_tokenizer,
    nepali_config["seq_len"]
)

nepali_test_dataset = NepaliReasoningDataset(
    nepali_test_records,
    nepali_tokenizer,
    nepali_config["seq_len"]
)

nepali_train_loader = DataLoader(
    nepali_train_dataset,
    batch_size=nepali_config["batch_size"],
    shuffle=True,
    drop_last=True
)

nepali_val_loader = DataLoader(
    nepali_val_dataset,
    batch_size=nepali_config["batch_size"],
    shuffle=False
)

nepali_test_loader = DataLoader(
    nepali_test_dataset,
    batch_size=nepali_config["batch_size"],
    shuffle=False
)

print("Nepali custom tokenizer loaded.")
print("Train batches:", len(nepali_train_loader))
print("Validation batches:", len(nepali_val_loader))
print("Test batches:", len(nepali_test_loader))

print("\nTokenizer test:")
sample = "नेपाल एक सुन्दर देश हो।"
encoded = nepali_tokenizer.encode(sample)
print("Encoded length:", len(encoded))
print("Decoded:", nepali_tokenizer.decode(encoded))

NameError: name 'nepali_config' is not defined

In [ ]:
import os
import math
import json
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from tqdm.auto import tqdm

# Fine-tuning settings
EPOCHS = 3
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0

output_dir = "/kaggle/working/reasoning_project/nepali"
final_model_path = os.path.join(
    output_dir,
    "nepali_reasoning_finetuned.pt"
)
metrics_path = os.path.join(
    output_dir,
    "nepali_finetuning_metrics.json"
)

optimizer = AdamW(
    nepali_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

history = {
    "train_loss": [],
    "validation_loss": [],
    "validation_perplexity": []
}


def evaluate_model(model, loader):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            targets = batch["targets"].to(device)

            logits = model(input_ids)

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
                ignore_index=-100,
                reduction="sum"
            )

            valid_tokens = (targets != -100).sum().item()

            total_loss += loss.item()
            total_tokens += valid_tokens

    average_loss = total_loss / max(total_tokens, 1)
    perplexity = math.exp(min(average_loss, 20))

    return average_loss, perplexity


for epoch in range(EPOCHS):
    nepali_model.train()

    running_loss = 0.0
    running_tokens = 0

    progress_bar = tqdm(
        nepali_train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        targets = batch["targets"].to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = nepali_model(input_ids)

        loss = F.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            targets.reshape(-1),
            ignore_index=-100
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            nepali_model.parameters(),
            GRAD_CLIP
        )

        optimizer.step()

        valid_tokens = (targets != -100).sum().item()
        running_loss += loss.item() * valid_tokens
        running_tokens += valid_tokens

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = running_loss / max(running_tokens, 1)

    val_loss, val_perplexity = evaluate_model(
        nepali_model,
        nepali_val_loader
    )

    history["train_loss"].append(train_loss)
    history["validation_loss"].append(val_loss)
    history["validation_perplexity"].append(val_perplexity)

    print(
        f"\nEpoch {epoch + 1}/{EPOCHS}"
        f" | Train Loss: {train_loss:.4f}"
        f" | Val Loss: {val_loss:.4f}"
        f" | Val Perplexity: {val_perplexity:.2f}"
    )

# Save fine-tuned checkpoint
torch.save(
    {
        "model_state_dict": nepali_model.state_dict(),
        "config": nepali_config,
        "history": history,
        "epochs": EPOCHS,
        "learning_rate": LEARNING_RATE
    },
    final_model_path
)

with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

print("\nNepali fine-tuning completed.")
print("Model saved to:", final_model_path)
print("Metrics saved to:", metrics_path)

In [ ]:
import math
import torch
import torch.nn.functional as F

def evaluate_loader(model, loader):
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            targets = batch["targets"].to(device)

            logits = model(input_ids)

            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
                ignore_index=-100,
                reduction="sum"
            )

            valid_tokens = (targets != -100).sum().item()

            total_loss += loss.item()
            total_tokens += valid_tokens

    average_loss = total_loss / max(total_tokens, 1)
    perplexity = math.exp(min(average_loss, 20))

    return average_loss, perplexity


nepali_test_loss, nepali_test_perplexity = evaluate_loader(
    nepali_model,
    nepali_test_loader
)

print(f"Nepali Test Loss: {nepali_test_loss:.4f}")
print(f"Nepali Test Perplexity: {nepali_test_perplexity:.2f}")

In [ ]:
def generate_nepali_text(
    model,
    prompt,
    max_new_tokens=120,
    temperature=0.7
):
    model.eval()

    token_ids = nepali_tokenizer.encode(prompt)

    generated_ids = token_ids.copy()

    with torch.no_grad():
        for _ in range(max_new_tokens):
            input_ids = torch.tensor(
                [generated_ids],
                dtype=torch.long,
                device=device
            )

            outputs = model(input_ids)

            # Support models returning logits directly or as a tuple
            if isinstance(outputs, tuple):
                logits = outputs[0]
            elif hasattr(outputs, "logits"):
                logits = outputs.logits
            else:
                logits = outputs

            next_token_logits = logits[:, -1, :]

            # Temperature sampling
            next_token_logits = next_token_logits / temperature
            probabilities = torch.softmax(
                next_token_logits,
                dim=-1
            )

            next_token = torch.multinomial(
                probabilities,
                num_samples=1
            ).item()

            generated_ids.append(next_token)

            # Stop if EOS token is available
            if hasattr(nepali_tokenizer, "eos_token_id"):
                if next_token == nepali_tokenizer.eos_token_id:
                    break

    return nepali_tokenizer.decode(generated_ids)


for i, prompt in enumerate(nepali_prompts, 1):
    print(f"\n{'=' * 60}")
    print(f"Example {i}")
    print(generate_nepali_text(nepali_model, prompt))